In [ ]:
# ---------------
# Dependencies
# ---------------

import torch
import random
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

import pandas
import numpy
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, classification_report, average_precision_score

In [ ]:
# ------------------
# Reproducibility
# ------------------

def set_seed(seed: int):
    random.seed(seed)
    numpy.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True,warn_only=True)

seed = 50
set_seed(seed)

In [ ]:
# ------------------------------------
# Device Setup (Is CUDA available?)
# ------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device: ", DEVICE)

In [ ]:
# -----------------------------------
# Load Dataset from GDrive (Colab)
# -----------------------------------

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---------------
# Load dataset
# ---------------

df = pandas.read_csv("/content/drive/MyDrive/data/dataset.csv")

if "hash" in df.columns:
  df = df.drop(columns=["hash"])

# Balanced Dataset:
# df_majority = df[df["malware"] == 1]
# df_minority = df[df["malware"] == 0]

# df_majority_down = df_majority.sample(n=len(df_minority), random_state=42)
# df_balanced = pandas.concat([df_majority_down, df_minority]).sample(frac=1, random_state=42)

# df = df_balanced

In [ ]:
# -------------
# Data Setup
# -------------

X = df.drop(columns=['malware']).values.astype(numpy.float32)
y = df['malware'].values.astype(numpy.int64)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.30, stratify=df['malware'], random_state=42
)

X_train = torch.tensor(X_train).to(DEVICE)
X_val = torch.tensor(X_val).to(DEVICE)

y_train = torch.tensor(y_train).to(DEVICE)
y_val = torch.tensor(y_val).to(DEVICE)

In [ ]:
# ----------------------------------------------------------
# Create training data loader with weighted class samples
# ----------------------------------------------------------

# class_sample_counts = numpy.bincount(y_train.cpu().numpy())
# weights = 1. / class_sample_counts
# sample_weights = weights[y_train.cpu().numpy()]
# sample_weights = torch.from_numpy(sample_weights).float()

# sampler = WeightedRandomSampler(
#     sample_weights,
#     num_samples=len(sample_weights),
#     replacement=True
# )

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=128,
    # sampler=sampler
)

In [ ]:
VOCAB_SIZE = 307
SEQ_LEN = 100
NUM_CLASSES = 2
LATENT_DIM = 128
EMB_DIM = 128
BATCH_SIZE = 128
EPOCHS = 100

In [ ]:
class Discriminator(nn.Module):
  def __init__(self):
    super().__init__()

    self.embedding = nn.Embedding(VOCAB_SIZE, EMB_DIM)

    self.conv = nn.Sequential(
        nn.Conv1d(EMB_DIM, 128, 5, padding=2),
        nn.LeakyReLU(0.2),
        nn.Conv1d(128, 256, 5, padding=2),
        nn.LeakyReLU(0.2),
        nn.AdaptiveMaxPool1d(1)
    )

    self.fc = nn.Sequential(
        nn.Flatten(),
        nn.Linear(256, 128),
        nn.LeakyReLU(0.2),
        nn.Dropout(0.4),
        nn.Linear(128, NUM_CLASSES + 1)
    )

  def forward(self, x, embedded=False, return_features=False):
    if not embedded:
        x = x.long()
        x = self.embedding(x)

    x = x.permute(0, 2, 1) # (BatchSize, EmbeddingSpace, SequenceLength)
    features = self.conv(x)
    logits = self.fc(features)

    if return_features:
        return logits, features

    return logits

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()

        self.init_fc = nn.Linear(LATENT_DIM, 256)

        self.rnn = nn.GRU(
            input_size=EMB_DIM,
            hidden_size=256,
            batch_first=True
        )

        self.token_proj = nn.Linear(256, VOCAB_SIZE)
        self.start_token = nn.Parameter(torch.zeros(1, 1, EMB_DIM)) # this is the learned start token

    def forward(self, z, temperature=0.5):
        batch_size = z.size(0)
        h0 = torch.tanh(self.init_fc(z)).unsqueeze(0)
        inputs = self.start_token.repeat(batch_size, SEQ_LEN, 1)
        outputs, _ = self.rnn(inputs, h0)
        logits = self.token_proj(outputs)
        return F.gumbel_softmax(logits, tau=temperature, hard=True)

In [ ]:
D = Discriminator().to(DEVICE)
G = Generator().to(DEVICE)

optimizer_D = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
optimizer_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))

In [ ]:
# best_f1 = 0
# patience = 30
# patience_counter = 0

history = {
    "epoch": [],
    "d_loss": [],
    "g_loss": [],
}

for epoch in range(EPOCHS):

  D.train()
  G.train()

  for real_x, real_y in train_loader:

      real_x = real_x.to(DEVICE)
      real_y = real_y.to(DEVICE)
      batch_size = real_x.size(0)

      """Discriminator Training"""
      optimizer_D.zero_grad()

      # Real
      logits_real = D(real_x)
      loss_real = F.cross_entropy(
          logits_real,
          real_y
      )

      # Fake
      z = torch.randn(batch_size, LATENT_DIM).to(DEVICE)
      fake_probs = G(z)

      # Soft tokens to embedding space
      fake_emb = torch.matmul(
          fake_probs,
          D.embedding.weight
      )

      logits_fake = D(fake_emb, embedded=True)

      fake_labels = torch.full(
          (batch_size,),
          NUM_CLASSES,
          device=DEVICE
      )

      loss_fake = F.cross_entropy(
          logits_fake,
          fake_labels
      )

      loss_D = loss_real + loss_fake
      loss_D.backward()
      optimizer_D.step()

      """Generator Training"""
      optimizer_G.zero_grad()

      z = torch.randn(batch_size, LATENT_DIM).to(DEVICE)
      fake_probs = G(z)

      fake_emb = torch.matmul(
          fake_probs,
          D.embedding.weight
      )

      # Get features from discriminator
      logits_fake, feat_fake = D(fake_emb, embedded=True, return_features=True)
      _, feat_real = D(real_x, return_features=True)

      # Feature matching loss
      loss_G = F.mse_loss(
          feat_fake.mean(dim=0),
          feat_real.mean(dim=0)
      )

      loss_G.backward()
      optimizer_G.step()



  # """Validation"""
  # D.eval()
  # with torch.no_grad():

  #     logits = D(X_val)
  #     probs = F.softmax(logits[:, :NUM_CLASSES], dim=1)
  #     preds = torch.argmax(probs, dim=1)

  #     val_f1 = f1_score(
  #         y_val.cpu().numpy(),
  #         preds.cpu().numpy(),
  #         average="macro"
  #     )

  #     val_auc = roc_auc_score(
  #         y_val.cpu().numpy(),
  #         probs[:,1].cpu().numpy()
  #     )

  print(f"Epoch {epoch+1} | D {loss_D:.4f} | G {loss_G:.4f}") #| Macro-F1 {val_f1:.4f} | AUC {val_auc:.4f})

  # if val_f1 > best_f1:
  #     best_f1 = val_f1
  #     patience_counter = 0
  #     torch.save(D.state_dict(), "best_sgan.pt")
  # else:
  #     patience_counter += 1

  # if patience_counter > patience:
  #     print("Early stopping.")
  #     break

  history["epoch"].append(epoch + 1)
  history["d_loss"].append(loss_D.item())
  history["g_loss"].append(loss_G.item())

  # history["val_macro_f1"].append(val_f1)
  # history["val_auc"].append(val_auc)

history_df = pandas.DataFrame(history)
history_df.to_csv(f"history_seed_{seed}.csv", index=False)

In [ ]:
torch.save(D.state_dict(), f"sgan_seed-{seed}.pt")

In [ ]:
D.load_state_dict(torch.load(f"sgan_seed-{seed}.pt"))
D.eval()

with torch.no_grad():
    logits = D(X_val)
    probs = F.softmax(logits[:, :NUM_CLASSES], dim=1)
    preds = torch.argmax(probs, dim=1)

print(classification_report(
    y_val.cpu().numpy(),
    preds.cpu().numpy(),
    digits=4
))

print("PR-AUC:",
      average_precision_score(
          y_val.cpu().numpy(),
          probs[:,1].cpu().numpy()
      ))

print("ROC-AUC:",
      roc_auc_score(
          y_val.cpu().numpy(),
          probs[:,1].cpu().numpy()
      ))

### **Results Compilation**
___

#### SGAN (G+D)

**Seed 10**
```
              precision    recall  f1-score   support

           0     0.9567    0.8179    0.8819       324
           1     0.9954    0.9991    0.9972     12839

    accuracy                         0.9946     13163
   macro avg     0.9761    0.9085    0.9396     13163
weighted avg     0.9945    0.9946    0.9944     13163

PR-AUC: 0.9998515452768015
ROC-AUC: 0.9943322284820845
```

**Seed 20**
```
             precision    recall  f1-score   support

           0     0.9496    0.8148    0.8771       324
           1     0.9953    0.9989    0.9971     12839

    accuracy                         0.9944     13163
   macro avg     0.9725    0.9069    0.9371     13163
weighted avg     0.9942    0.9944    0.9942     13163

PR-AUC: 0.9998498224250775
ROC-AUC: 0.9943135979399187
```

**Seed 30**
```
              precision    recall  f1-score   support

           0     0.9745    0.8272    0.8948       324
           1     0.9957    0.9995    0.9976     12839

    accuracy                         0.9952     13163
   macro avg     0.9851    0.9133    0.9462     13163
weighted avg     0.9951    0.9952    0.9950     13163

PR-AUC: 0.9998538040395232
ROC-AUC: 0.9945331979433804
```

**Seed 40**
```
              precision    recall  f1-score   support

           0     0.9354    0.8488    0.8900       324
           1     0.9962    0.9985    0.9974     12839

    accuracy                         0.9948     13163
   macro avg     0.9658    0.9236    0.9437     13163
weighted avg     0.9947    0.9948    0.9947     13163

PR-AUC: 0.9997291102862584
ROC-AUC: 0.9920657929783772
```

**Seed 50**
```
              precision    recall  f1-score   support

           0     0.9547    0.8457    0.8969       324
           1     0.9961    0.9990    0.9976     12839

    accuracy                         0.9952     13163
   macro avg     0.9754    0.9223    0.9472     13163
weighted avg     0.9951    0.9952    0.9951     13163

PR-AUC: 0.9998660764521053
ROC-AUC: 0.9949084531217096
```